# Tutorial: Fictional Crime Scenario Replay

This notebook uses a fully fictional, low-severity campus property incident to show how Knoema can replay a scenario, collect agent actions, and compare the run against reference milestones.


## Audience and Goal

Audience: public-safety researchers, simulation evaluators, and collaborators who need a cautious replay workflow.

Prerequisites: basic Python and the repository installed with `pip install -e .`.

By the end, you can build an anonymized replay fixture, run deterministic agents, and inspect whether the simulated trace covers expected milestones. This is not a crime prediction, suspect scoring, or operational decision tool.


## Outline

1. Import Knoema and define the fictional replay scope.
2. Create personas for a campus property incident.
3. Schedule reference events and run a one-day replay.
4. Compare action logs with milestone coverage.
5. Store and retrieve replay notes from long-term memory.
6. Try one small exercise.


## 1. Setup

The notebook uses `LocalClient` so execution is deterministic and does not require API keys.


In [ ]:
import importlib.util
import json
import subprocess
import sys
from collections import Counter
from datetime import datetime, timedelta

if importlib.util.find_spec('knoema') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])

from knoema import (
    Environment,
    LocalClient,
    Memory,
    Persona,
    Personality,
    Simulator,
    SQLiteFaissMemoryStore,
    WorldEvent,
)


## 2. Fictional Replay Fixture

The case is synthetic: a missing drone battery at a fictional training campus. The milestone list intentionally avoids suspect labels and focuses on accountable process steps.


In [ ]:
start_time = datetime(2026, 4, 6, 8, 0)
location = 'Harborview Training Campus > Robotics Lab'

reference_events = [
    WorldEvent(
        timestamp=start_time + timedelta(hours=1),
        event_type='case.report',
        participants=['mara'],
        location=location,
        description='Mara reports a missing drone battery from the robotics shelf.',
    ),
    WorldEvent(
        timestamp=start_time + timedelta(hours=3),
        event_type='case.access_log_ready',
        participants=['theo'],
        location=location,
        description='Door and camera logs are ready for neutral review.',
    ),
    WorldEvent(
        timestamp=start_time + timedelta(hours=6),
        event_type='case.item_found',
        participants=['mara', 'theo'],
        location=location,
        description='The drone battery is found on a maintenance cart after room cleanup.',
    ),
]

milestone_terms = {
    'report missing item': ['missing battery', 'report'],
    'preserve neutral evidence': ['evidence list', 'neutral'],
    'review access logs': ['access log', 'camera'],
    'reconstruct checkout timeline': ['checkout timeline'],
    'recover misplaced item': ['recover', 'maintenance cart'],
    'close without suspect label': ['closeout', 'no suspect label'],
}

[(event.timestamp.isoformat(), event.event_type, event.description) for event in reference_events]


## 3. Personas

The agents represent roles, not real people. Their job is to generate a trace that can be audited against the reference milestones.


In [ ]:
mara = Persona(
    agent_id='mara',
    name='Mara Lee',
    age=29,
    background='Robotics lab coordinator at a fictional training campus.',
    personality=Personality(
        openness=0.58,
        conscientiousness=0.86,
        extraversion=0.42,
        agreeableness=0.74,
        neuroticism=0.34,
    ),
    values=['accuracy', 'student safety', 'neutral documentation'],
    goals=['recover the missing battery', 'avoid unsupported accusations'],
)

theo = Persona(
    agent_id='theo',
    name='Theo Grant',
    age=35,
    background='Campus safety coordinator trained to keep low-severity incidents procedural.',
    personality=Personality(
        openness=0.49,
        conscientiousness=0.91,
        extraversion=0.48,
        agreeableness=0.66,
        neuroticism=0.24,
    ),
    values=['fair process', 'privacy', 'clear chain of custody'],
    goals=['review access logs', 'close the case only with evidence'],
)

juno = Persona(
    agent_id='juno',
    name='Juno Park',
    age=21,
    background='Robotics club mentor who helps reconstruct equipment checkout history.',
    personality=Personality(
        openness=0.77,
        conscientiousness=0.69,
        extraversion=0.56,
        agreeableness=0.82,
        neuroticism=0.31,
    ),
    values=['helpfulness', 'transparency', 'repairing trust'],
    goals=['reconstruct the checkout timeline', 'keep club members from being mislabeled'],
)

agents = [mara, theo, juno]
[agent.agent_id for agent in agents]


## 4. Deterministic Replay Responder

The responder returns strict JSON actions. It is intentionally simple so differences in replay output come from the simulation state, not a remote model.


In [ ]:
turns = Counter()


def replay_responder(messages):
    system_prompt = messages[0]['content']
    agent_id = next(agent.agent_id for agent in agents if f'Persona ID: {agent.agent_id}' in system_prompt)
    turns[agent_id] += 1
    turn = turns[agent_id]

    scripts = {
        'mara': [
            'I record the missing battery report and start a neutral evidence list.',
            'I compare the shelf photo with the maintenance cart route.',
            'I confirm we can closeout the case with no suspect label.',
        ],
        'theo': [
            'I review the access log and camera summary before interviewing anyone.',
            'I ask for the checkout timeline and keep the review procedural.',
            'I recover the misplaced battery from the maintenance cart and document custody.',
        ],
        'juno': [
            'I reconstruct the checkout timeline from the club sign-out sheet.',
            'I remind the team that neutral notes protect uninvolved students.',
            'I help write a closeout note with no suspect label.',
        ],
    }
    content = scripts[agent_id][min(turn - 1, len(scripts[agent_id]) - 1)]
    target = {'mara': 'theo', 'theo': 'mara', 'juno': 'mara'}[agent_id]
    return json.dumps(
        {'action_type': 'document', 'target': target, 'content': content},
        ensure_ascii=False,
    )

local_llm = LocalClient(replay_responder)


## 5. Run the Replay

The scheduler injects the fictional reference events as the simulated day advances.


In [ ]:
environment = Environment(
    start_time=start_time,
    location_path=('Fictional City', 'Harborview Training Campus', 'Robotics Lab'),
    conditions={
        'case_type': 'low_severity_property_incident',
        'data_policy': 'synthetic_and_anonymized',
        'use_for_prediction': False,
    },
)

sim = Simulator(
    agents=agents,
    environment=environment,
    tick_duration_minutes=120,
    llm=local_llm,
)
for event in reference_events:
    sim.scheduler.schedule(event)

logs = sim.run(duration_days=1)
len(logs), logs[0].timestamp.isoformat(), logs[-1].timestamp.isoformat()


## 6. Inspect the Trace

A small sample is enough to confirm the run is producing auditable action records.


In [ ]:
trace_sample = [
    {
        'tick': entry.tick,
        'time': entry.timestamp.isoformat(),
        'agent': entry.agent_id,
        'target': entry.action.target,
        'content': entry.action.content,
    }
    for entry in logs[:9]
]
trace_sample


## 7. Milestone Coverage

The score is a replay sanity check: did the generated trace mention the expected process milestones? It does not validate truth, guilt, or operational readiness.


In [ ]:
action_texts = [entry.action.content.lower() for entry in logs]


def covered(terms):
    return any(all(term in action_text for term in terms) for action_text in action_texts)


coverage = {name: covered(terms) for name, terms in milestone_terms.items()}
coverage_score = sum(coverage.values()) / len(coverage)

{'score': round(coverage_score, 2), 'coverage': coverage}


In [ ]:
assert coverage_score >= 0.8


## 8. Store Replay Notes

Long-term memory can keep compact replay notes and retrieve them by semantic query for later review.


In [ ]:
store = SQLiteFaissMemoryStore(':memory:')
for index, entry in enumerate(logs[:12]):
    store.add(
        Memory(
            id=f'replay-note-{index}',
            agent_id=entry.agent_id,
            timestamp=entry.timestamp,
            content=entry.action.content,
            memory_type='episodic',
            importance=0.7,
        )
    )

retrieved = store.retrieve('recover missing battery maintenance cart no suspect label', k=4)
[memory.content for memory in retrieved]


In [ ]:
store.close()


## Exercise

Add a fourth reference event where a second item is reported missing, then rerun the replay. Update `milestone_terms` with one new neutral process milestone and check whether `coverage_score` still passes.

Answer scaffold: add a `WorldEvent(...)` to `reference_events`, add one entry to `milestone_terms`, then update `replay_responder` so at least one agent documents that milestone.


## Pitfall and Extension

Pitfall: using real incidents or identifiable people turns a demo into a privacy and fairness risk. Keep replay fixtures synthetic unless there is explicit authorization and a data governance process.

Extension: export `logs` to JSONL and replay the same trace in a dashboard or game engine adapter.
